# Asistente Experto en Gastronomía
## Proyecto Final — IA Generativa con Gemini, RAG y LangGraph

Para este proyecto he construido un agente de IA especializado en gastronomía.
El agente responde preguntas usando una base de conocimiento propia que yo mismo he creado,
combinando técnicas de RAG, LangGraph y memoria de conversación.

He estructurado el desarrollo en los siguientes pasos:

1. Configuración del entorno y carga de la API key
2. Creación de la base de conocimiento vectorial con ChromaDB
3. Diseño del system prompt
4. Construcción del agente con LangGraph
5. Implementación de la memoria de conversación
6. Pruebas con preguntas de ejemplo
7. Chat interactivo

---
## PASO 1 — Configuración del entorno

Lo primero que hago es cargar las librerías necesarias y la API key de Gemini.
He guardado la key en un archivo `.env` para no exponerla en el código,
ya que si subo el notebook a GitHub la clave quedaría visible para cualquiera.

In [24]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
print(f"Key cargada: {GEMINI_API_KEY[:8]}...")

Key cargada: AIzaSyBa...


---
## PASO 2 — Creación de la base de conocimiento

He elegido **gastronomía** como dominio del asistente porque es un área amplia que permite
cubrir temas muy distintos: cocinas del mundo, técnicas culinarias, repostería y maridaje.

He creado 6 documentos con contenido propio sobre los siguientes temas:
paella valenciana, sushi, técnicas francesas, chocolate, maridaje de vinos y cocina molecular.

In [25]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

documentos = [
    Document(
        page_content="""La paella valenciana es uno de los platos más representativos de España.
        Se elabora con arroz de grano corto, preferiblemente la variedad Bomba o Senia.
        Los ingredientes tradicionales incluyen pollo, conejo, judías verdes (ferradura),
        garrofó (alubia blanca grande), tomate, aceite de oliva, pimentón dulce, azafrán y agua.
        La paella se cocina en una sartén plana y ancha llamada paellera.
        El secreto está en el socarrat: la capa crujiente de arroz que se forma en el fondo.
        Nunca se añade chorizo, cebolla ni limón a una paella valenciana auténtica.""",
        metadata={"fuente": "gastronomia_española", "tema": "arroces"}
    ),
    Document(
        page_content="""El sushi es la joya de la gastronomía japonesa. Consiste en arroz
        avinagrado (shari) combinado con diversos ingredientes como pescado crudo, mariscos,
        verduras o tortilla japonesa (tamagoyaki). Los tipos más comunes son:
        - Nigiri: bola de arroz con pescado encima, moldeada a mano.
        - Maki: rollo de arroz y relleno envuelto en alga nori.
        - Sashimi: solo pescado crudo, sin arroz.
        - Temaki: cono de alga nori relleno de arroz e ingredientes.
        El wasabi y el jengibre encurtido (gari) se sirven como acompañamientos.
        La salsa de soja se usa para mojar ligeramente, nunca para empapar el arroz.""",
        metadata={"fuente": "gastronomia_japonesa", "tema": "sushi"}
    ),
    Document(
        page_content="""Las técnicas básicas de cocina francesa son la base de la cocina moderna.
        El mise en place es la preparación y organización de todos los ingredientes antes de cocinar.
        El salteado (sauté) consiste en cocinar a fuego alto con poca grasa y movimiento constante.
        El braseado combina dorado inicial y cocción lenta en líquido, ideal para carnes duras.
        La emulsión es la mezcla estable de dos líquidos no miscibles, como en la mayonesa o la
        salsa holandesa. El escaldado (blanching) consiste en sumergir brevemente en agua hirviendo
        y enfriar en agua con hielo para preservar color y textura de verduras.
        Las cinco salsas madre francesas son: bechamel, velouté, española, holandesa y tomate.""",
        metadata={"fuente": "tecnicas_cocina", "tema": "tecnicas_basicas"}
    ),
    Document(
        page_content="""El chocolate tiene su origen en Mesoamérica, donde los mayas y aztecas
        consumían el cacao como bebida amarga. Llega a Europa en el siglo XVI.
        Los tipos principales de chocolate son:
        - Negro (amargo): mínimo 70% cacao, sin leche, sabor intenso.
        - Con leche: entre 25-45% cacao, más dulce y cremoso.
        - Blanco: manteca de cacao, azúcar y leche, sin sólidos de cacao.
        - Ruby: chocolate rosa natural, hecho con granos de cacao ruby.
        El temperado del chocolate es el proceso de calentar y enfriar controladamente
        para obtener un chocolate con brillo, crujido y textura perfecta.
        El maridaje del chocolate negro con vino tinto es muy apreciado en gastronomía.""",
        metadata={"fuente": "reposteria", "tema": "chocolate"}
    ),
    Document(
        page_content="""El maridaje es el arte de combinar vinos con platos para que se potencien
        mutuamente. Principios básicos:
        - Vinos blancos ligeros (Albariño, Verdejo) van bien con mariscos y pescados blancos.
        - Vinos tintos con cuerpo (Rioja, Ribera del Duero) maridan con carnes rojas y guisos.
        - Vinos espumosos (Cava, Champagne) son versátiles, van bien con aperitivos y sushi.
        - Vinos dulces (Pedro Ximénez, Moscatel) acompañan postres y quesos azules.
        La regla de oro es equilibrio: un plato delicado pide vino delicado,
        un plato potente pide vino potente.
        La temperatura de servicio es crucial: los blancos entre 8-12°C, los tintos entre 16-18°C.""",
        metadata={"fuente": "enologia", "tema": "maridaje"}
    ),
    Document(
        page_content="""La gastronomía molecular, también llamada cocina de vanguardia, aplica
        ciencia y tecnología a la cocina. Fue popularizada por Ferran Adrià en elBulli y
        Heston Blumenthal en The Fat Duck.
        Técnicas principales:
        - Esferificación: crear esferas con interior líquido usando alginato y cloruro cálcico.
        - Gelificación con agar-agar: gelatinas que no se derriten a temperatura ambiente.
        - Espumas: usar lecitina de soja o proespuma para crear texturas aéreas.
        - Nitrógeno líquido: congelar al instante a -196°C para texturas únicas.
        - Sous vide: cocción al vacío a temperatura precisa durante horas para texturas perfectas.
        El objetivo no es la tecnología en sí, sino crear nuevas experiencias gastronómicas.""",
        metadata={"fuente": "cocina_molecular", "tema": "tecnicas_avanzadas"}
    ),
]

print(f"Documentos preparados: {len(documentos)}")
for i, doc in enumerate(documentos):
    print(f"  Doc {i+1}: {doc.metadata['tema']} ({doc.metadata['fuente']})")

Documentos preparados: 6
  Doc 1: arroces (gastronomia_española)
  Doc 2: sushi (gastronomia_japonesa)
  Doc 3: tecnicas_basicas (tecnicas_cocina)
  Doc 4: chocolate (reposteria)
  Doc 5: maridaje (enologia)
  Doc 6: tecnicas_avanzadas (cocina_molecular)


In [26]:
import requests
import shutil
from langchain_core.embeddings import Embeddings

class GeminiEmbeddings(Embeddings):
    def __init__(self, api_key, model="models/gemini-embedding-001"):
        self.api_key = api_key
        self.model = model
        self.url = f"https://generativelanguage.googleapis.com/v1beta/{model}:embedContent"

    def _embed(self, text):
        response = requests.post(
            self.url,
            params={"key": self.api_key},
            json={"model": self.model, "content": {"parts": [{"text": text}]}}
        )
        response.raise_for_status()
        return response.json()["embedding"]["values"]

    def embed_documents(self, texts):
        return [self._embed(t) for t in texts]

    def embed_query(self, text):
        return self._embed(text)


# Cierro la conexión anterior antes de borrar (necesario en Windows)
if "vectorstore" in dir():
    try:
        vectorstore._client.close()
    except Exception:
        pass

if os.path.exists("./chroma_gastronomia"):
    shutil.rmtree("./chroma_gastronomia")

embeddings = GeminiEmbeddings(api_key=GEMINI_API_KEY)

vectorstore = Chroma.from_documents(
    documents=documentos,
    embedding=embeddings,
    persist_directory="./chroma_gastronomia",
    collection_name="gastronomia"
)

print("Base de conocimiento creada en ChromaDB")
print(f"Documentos indexados: {vectorstore._collection.count()}")

print("\nPrueba de búsqueda semántica:")
resultados = vectorstore.similarity_search("¿cómo se hace el arroz?", k=2)
for r in resultados:
    print(f"  -> Documento recuperado: {r.metadata['tema']}")

Base de conocimiento creada en ChromaDB
Documentos indexados: 6

Prueba de búsqueda semántica:
  -> Documento recuperado: arroces
  -> Documento recuperado: sushi


---
## PASO 3 — Diseño del system prompt

He diseñado el system prompt para que el agente actúe como un chef experto con un tono
profesional pero cercano. Las decisiones que tomé son:

- **Rol claro**: Chef AI, experto en gastronomía internacional
- **Tono**: como un chef experimentado hablando con un alumno, preciso pero accesible
- **Límite explícito**: si no tiene información, lo dice. No quiero que invente.
- **Restricción de dominio**: solo quiero que responda sobre gastronomía
- **Inyección de contexto**: el prompt incluye un placeholder `{context}` donde se insertan
  los fragmentos recuperados de ChromaDB en cada consulta

In [27]:
SYSTEM_PROMPT = """Eres Chef AI, un asistente experto en gastronomía con profundo conocimiento
en cocina internacional, técnicas culinarias, ingredientes, maridajes y cultura gastronómica.

Tu forma de responder:
- Usa un tono profesional pero accesible, como un chef experimentado hablando con un alumno.
- Sé preciso y útil. Cuando des recetas o técnicas, sé específico con cantidades y pasos.
- Si el contexto proporcionado contiene la respuesta, úsalo como base principal.
- Si no tienes información suficiente sobre algo, dilo claramente: no inventes.
- Puedes añadir curiosidades o consejos relacionados cuando sea relevante.

Tu especialidad:
- Cocinas del mundo (española, japonesa, francesa, italiana, etc.)
- Técnicas culinarias (básicas y de vanguardia)
- Repostería y chocolatería
- Maridaje de vinos y bebidas
- Historia y cultura de los alimentos

Límites:
- Solo respondes sobre gastronomía y temas relacionados con la alimentación.
- Si te preguntan algo fuera de tu área, redirige amablemente hacia temas gastronómicos.

Contexto recuperado de la base de conocimiento:
{context}
"""

print("System prompt definido")
print(f"Longitud: {len(SYSTEM_PROMPT)} caracteres")

System prompt definido
Longitud: 1077 caracteres


---
## PASO 4 — Construcción del agente con LangGraph

He implementado el agente usando LangGraph, que organiza la lógica como un **grafo de nodos**.
Cada nodo es una función que transforma el estado del agente.

Mi grafo tiene dos nodos y sigue este flujo:

```
Usuario pregunta
       ↓
[Nodo 1: recuperar] → busca en ChromaDB los 3 fragmentos más relevantes
       ↓
[Nodo 2: generar]   → Gemini genera la respuesta con ese contexto + historial
       ↓
Respuesta al usuario
```

He definido `AgentState` como el objeto que viaja entre nodos y acumula toda la información:
la pregunta actual, el contexto recuperado y el historial de mensajes (memoria).

In [28]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    context: str
    question: str

# --- LLM activo: Gemma 3 (local con Ollama, sin cuotas) ---
# Requisito previo: instalar Ollama (https://ollama.com) y ejecutar:
#   ollama pull gemma3:4b
llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.3
)

# --- LLM alternativo: Gemini (descomentar cuando se recupere la cuota) ---
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.0-flash",
#     google_api_key=GEMINI_API_KEY,
#     temperature=0.3
# )

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("LLM y retriever configurados (usando Gemma 3 local)")

LLM y retriever configurados (usando Gemma 3 local)


In [29]:
# Nodo 1: recupera el contexto relevante de ChromaDB
def recuperar_contexto(state: AgentState) -> AgentState:
    pregunta = state["question"]
    docs = retriever.invoke(pregunta)
    contexto = "\n\n---\n\n".join([doc.page_content for doc in docs])
    return {"context": contexto}


# Nodo 2: genera la respuesta con Gemini usando el contexto y el historial
def generar_respuesta(state: AgentState) -> AgentState:
    contexto = state["context"]
    pregunta = state["question"]
    historial = state.get("messages", [])
    
    # Inyecto el contexto recuperado en el system prompt
    system_con_contexto = SYSTEM_PROMPT.format(context=contexto)
    
    # Construyo la lista de mensajes: sistema + historial + pregunta actual
    mensajes = [SystemMessage(content=system_con_contexto)]
    mensajes.extend(historial)
    mensajes.append(HumanMessage(content=pregunta))
    
    respuesta = llm.invoke(mensajes)
    
    # Añado al historial la pregunta y la respuesta para mantener la memoria
    return {
        "messages": [
            HumanMessage(content=pregunta),
            respuesta
        ]
    }


# Construyo el grafo conectando los nodos en orden
grafo = StateGraph(AgentState)
grafo.add_node("recuperar", recuperar_contexto)
grafo.add_node("generar", generar_respuesta)
grafo.set_entry_point("recuperar")
grafo.add_edge("recuperar", "generar")
grafo.add_edge("generar", END)

agente = grafo.compile()

print("Agente LangGraph compilado correctamente")

Agente LangGraph compilado correctamente


---
## PASO 5 — Memoria de conversación

He implementado la memoria acumulando los mensajes en la lista `messages` del estado.
En cada nueva pregunta, paso el historial completo al agente, que lo incluye en el
contexto enviado a Gemini. Así el modelo puede referirse a respuestas anteriores.

La función `preguntar()` abstrae este proceso: recibe la pregunta y el historial,
ejecuta el agente y devuelve la respuesta junto con el historial actualizado.

In [30]:
def preguntar(pregunta: str, historial: list) -> tuple:
    estado_inicial = {
        "question": pregunta,
        "messages": historial,
        "context": ""
    }
    
    resultado = agente.invoke(estado_inicial)
    
    respuesta_texto = resultado["messages"][-1].content
    historial_nuevo = resultado["messages"]
    
    return respuesta_texto, historial_nuevo


print("Función preguntar() lista")

Función preguntar() lista


---
## PASO 6 — Pruebas con preguntas de ejemplo

He preparado 5 preguntas para demostrar el funcionamiento del agente.
Las primeras 4 prueban la recuperación RAG sobre distintos documentos.
La quinta demuestra que la **memoria funciona**: hace referencia a temas
tratados en preguntas anteriores de la misma conversación.

In [31]:
historial = []

preguntas = [
    "¿Cuál es la diferencia entre un nigiri y un maki?",
    "¿Qué ingredientes lleva una paella valenciana auténtica?",
    "¿Qué es la esferificación en cocina molecular?",
    "¿Qué vino marida mejor con un pescado blanco?",
    "De todo lo que me has explicado, ¿qué técnica te parece más difícil de aprender en casa?",
]

for i, pregunta in enumerate(preguntas, 1):
    print(f"\n{'='*60}")
    print(f"PREGUNTA {i}: {pregunta}")
    print(f"{'='*60}")
    
    respuesta, historial = preguntar(pregunta, historial)
    
    print(f"RESPUESTA:\n{respuesta}")
    print(f"\n[Mensajes en memoria: {len(historial)}]")


PREGUNTA 1: ¿Cuál es la diferencia entre un nigiri y un maki?
RESPUESTA:
¡Excelente pregunta! Es una distinción fundamental en el mundo del sushi, y es muy común confundirlos al principio. Permíteme explicarte la diferencia entre nigiri y maki de forma clara y precisa:

**Nigiri:**

*   **Definición:** El nigiri es, en esencia, una "bola" de arroz avinagrado (shari) que se ha moldeado a mano. Sobre esta base de arroz, se coloca una porción de pescado crudo (o marisco, o incluso algún otro ingrediente) que ha sido cortado en una forma específica.
*   **Presentación:** La clave del nigiri es la presentación manual. El chef utiliza sus manos para darle forma al arroz y colocar el pescado encima, creando una unión visualmente atractiva.
*   **Sabor:** El sabor del nigiri se basa en la calidad del arroz y en el sabor del pescado que se utiliza. El arroz avinagrado proporciona un contraste de sabor con el pescado fresco.
*   **Ejemplo:** Imagina una bola de arroz con una fina capa de sal, s

---
## PASO 7 — Chat interactivo

Por último, he implementado una celda interactiva para poder conversar libremente
con el agente. El historial se mantiene durante toda la sesión.

He usado **ipywidgets** en lugar de `input()` para que el cuadro de texto funcione
correctamente con teclado en español (tildes, ñ, etc.) tanto en VS Code como en Jupyter.
Pulsa **Enviar** o **Enter** para mandar el mensaje. Escribe `salir` para terminar.

In [34]:
import ipywidgets as widgets
from IPython.display import display, clear_output

historial_chat = []

salida = widgets.Output()
texto = widgets.Text(
    placeholder="Escribe tu pregunta aquí...",
    layout=widgets.Layout(width="80%")
)
boton = widgets.Button(description="Enviar", button_style="primary")

def on_enviar(b):
    global historial_chat
    pregunta = texto.value.strip()
    texto.value = ""
    if not pregunta:
        return
    if pregunta.lower() in ["salir", "exit", "quit"]:
        with salida:
            print("¡Hasta pronto! Buen provecho.")
        boton.disabled = True
        texto.disabled = True
        return
    with salida:
        print(f"\nTú: {pregunta}")
        print("Chef AI: pensando...")
    respuesta, historial_chat = preguntar(pregunta, historial_chat)
    with salida:
        clear_output(wait=True)

    # Reimprime todo el historial limpio
    with salida:
        for i in range(0, len(historial_chat) - 1, 2):
            h = historial_chat[i].content
            r = historial_chat[i + 1].content if i + 1 < len(historial_chat) else ""
            print(f"Tú: {h}")
            print(f"Chef AI: {r}")
            print("-" * 60)

texto.on_submit(on_enviar)
boton.on_click(on_enviar)

print("Chef AI listo. Escribe tu pregunta y pulsa Enviar (o Enter).")
display(widgets.HBox([texto, boton]))
display(salida)

Chef AI listo. Escribe tu pregunta y pulsa Enviar (o Enter).


C:\Users\dani\AppData\Local\Temp\ipykernel_9420\1161664784.py:41: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  texto.on_submit(on_enviar)


Output()